## **Problem Statement**

### Business Context

The prices of the stocks of companies listed under a global exchange are influenced by a variety of factors, with the company's financial performance, innovations and collaborations, and market sentiment being factors that play a significant role. News and media reports can rapidly affect investor perceptions and, consequently, stock prices in the highly competitive financial industry. With the sheer volume of news and opinions from a wide variety of sources, investors and financial analysts often struggle to stay updated and accurately interpret its impact on the market. As a result, investment firms need sophisticated tools to analyze market sentiment and integrate this information into their investment strategies.

### Problem Definition

With an ever-rising number of news articles and opinions, an investment startup aims to leverage artificial intelligence to address the challenge of interpreting stock-related news and its impact on stock prices. They have collected historical daily news for a specific company listed under NASDAQ, along with data on its daily stock price and trade volumes.

As a member of the Data Science and AI team in the startup, you have been tasked with developing an AI-driven sentiment analysis system that will automatically process and analyze news articles to gauge market sentiment, and summarizing the news at a weekly level to enhance the accuracy of their stock price predictions and optimize investment strategies. This will empower their financial analysts with actionable insights, leading to more informed investment decisions and improved client outcomes.

### Data Dictionary

* `Date` : The date the news was released
* `News` : The content of news articles that could potentially affect the company's stock price
* `Open` : The stock price (in \$) at the beginning of the day
* `High` : The highest stock price (in \$) reached during the day
* `Low` :  The lowest stock price (in \$) reached during the day
* `Close` : The adjusted stock price (in \$) at the end of the day
* `Volume` : The number of shares traded during the day
* `Label` : The sentiment polarity of the news content
    * 1: positive
    * 0: neutral
    * -1: negative

## **Installing and Importing the necessary libraries**

In [ ]:
# Installing the sentence-transformers and gensim libraries for word embeddings
# Added matplotlib, seaborn, nltk, and tensorflow to ensure all dependencies are met
!pip install numpy==1.26.4 \
             scikit-learn==1.6.1 \
             scipy==1.13.1 \
             gensim==4.3.3 \
             sentence-transformers==3.4.1 \
             pandas==2.2.2 \
             matplotlib \
             seaborn \
             nltk \
             tensorflow


Note:
- After running the above cell, kindly restart the runtime (for Google Colab) or notebook kernel (for Jupyter Notebook), and run all cells sequentially from the next cell.
- On executing the above line of code, you might see a warning regarding package dependencies. This error message can be ignored as the above code ensures that all necessary libraries and their dependencies are maintained to successfully execute the code in this notebook.

In [ ]:
# to read and manipulate the data
import numpy as np
import pandas as pd
pd.set_option('max_colwidth', None)    # setting column to the maximum column width as per the data

# to visualise data
import matplotlib.pyplot as plt
import seaborn as sns

# To import Word2Vec
from gensim.models import Word2Vec

# Deep Learning library
import torch

# Clears previous session
import gc

# to load transformer models
from sentence_transformers import SentenceTransformer
from transformers import T5Tokenizer, T5ForConditionalGeneration, pipeline

# To split data into train and test sets
from sklearn.model_selection import train_test_split

from sklearn.preprocessing import LabelEncoder

# To build a Random Forest model
from sklearn.ensemble import RandomForestClassifier

# To compute metrics to evaluate the model
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score, confusion_matrix

# to ignore unnecessary warnings
import warnings
warnings.filterwarnings("ignore")

# Import TensorFlow and Keras for deep learning model building.
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras import regularizers

## **Loading the dataset**

In [ ]:

from google.colab import drive
drive.mount('/content/drive')

try:
    df = pd.read_csv("/content/drive/MyDrive/Gen AI/Project_1_StockMarketSentimentAnalysis/stock_news.csv")


    print("Data loaded successfully.")
except FileNotFoundError:
    print("File not found. Please ensure the file is uploaded.")

# creating a copy of the data
data = df.copy()

# Convert Date column to datetime objects
data['Date'] = pd.to_datetime(data['Date'], format='mixed')

# Double check the conversion worked
print(data['Date'].head())

## **Data Overview**

In [ ]:
# Display the first 5 rows
display(data.head(5))

# Check the shape of the dataset
print(f"\nShape of the dataset: {data.shape}")

# Check for data types
print("\nData Info:")
data.info()  # .info() prints automatically, no need for print()

# Check for missing values
print("\nMissing values:")
print(data.isnull().sum())

# Checking for duplicate values
print(f"\nDuplicate values: {data.duplicated().sum()}")

# Keeping only the first occurrence of duplicate values and dropping the rest
data = data.drop_duplicates(keep='first')

# Reset the index after dropping duplicates to maintain a clean, continuous index
data = data.reset_index(drop=True)

# Statistical summary of numerical columns
print("\nStatistical Summary:")
display(data.describe())  # Removed outer print() to avoid "None" output

# Check the distribution of the target variable 'Label'
print("\nClass Distribution:")
print(data['Label'].value_counts())

## **Exploratory Data Analysis**

### **Univariate Analysis**

* Distribution of individual variables
* Compute and check the distribution of the length of news content

In [ ]:
# Distribution of individual variables

# Check the balance of the target variable Label
plt.figure(figsize=(8, 5))
sns.countplot(x='Label', data=data, palette='coolwarm')
plt.title('Distribution of Sentiment Labels')
plt.xlabel('Sentiment (1: Positive, 0: Neutral, -1: Negative)')
plt.ylabel('Count')
plt.show()

# Print exact numbers
print(data['Label'].value_counts())

**Observation:** Class Imbalance

The dataset is heavily skewed. We have 270 Positive samples and 141 Negative samples, but only 7 Neutral samples.

**Business Insight:** Financial news in this dataset rarely captures "neutral" events; it focuses on significant positive or negative developments.

**Technical Impact:** The model will struggle to learn the "Neutral" class. We must use Weighted F1-Score rather than Accuracy to evaluate performance effectively.

In [ ]:
#Distribution of Stock Closing Price
plt.figure(figsize=(10, 5))
sns.histplot(data['Close'], bins=50, kde=True, color='green')
plt.title('Distribution of Closing Price')
plt.xlabel('Price ($)')
plt.show()

In [ ]:
# 4. Distribution of Trading Volume
plt.figure(figsize=(10, 5))
sns.histplot(data['Volume'], bins=50, kde=True, color='orange')
plt.title('Distribution of Trading Volume')
plt.xlabel('Volume')
plt.show()

**Observation:** Price and Volume

**Closing Price:** The stock price distribution appears relatively normal, centered around the mean. There are no extreme outliers that would require capping/flooring.

**Volume:** The trading volume distribution is right-skewed. Most days have moderate trading, but there are a few days with extremely high volume. These "spike" days likely correspond to major news events, which we will investigate in the Bivariate Analysis.

In [ ]:
# Compute and check the distribution of the length of news content

# Create a new column for the length of the news text
data['news_length'] = data['News'].apply(len)

# Display the statistics for news length
print("\nNews Length Statistics:")
print(data['news_length'].describe())

# Plot the distribution
plt.figure(figsize=(10, 5))
sns.histplot(data['news_length'], bins=30, kde=True, color='purple')
plt.title('Distribution of News Article Lengths')
plt.xlabel('Length (Number of Characters)')
plt.show()

**Observation: News Length**

The news length distribution shows that most text entries are short (likely headlines or brief snippets) rather than long articles.

**Technical Insight:** Since the text is short, complex context tracking (like in full documents) is less critical than capturing the immediate semantic meaning of key phrases. This supports using Sentence Transformers over simple word averaging.

**Recommendation:** We do not need to truncate or pad the text excessively during the embedding phase, as the lengths are relatively consistent and short.

### **Bivariate Analysis**

* Correlation
* Sentiment Polarity vs Price
* Date vs Price

**Note**: The above points are listed to provide guidance on how to approach bivariate analysis. Analysis has to be done beyond the above listed points to get maximum scores.

In [ ]:
# --- 1. Correlation Matrix ---
# Select numerical columns for correlation analysis
numerical_cols = ['Open', 'High', 'Low', 'Close', 'Volume', 'Label', 'news_length']
corr_matrix = data[numerical_cols].corr(numeric_only = True)

In [ ]:
# Plot Heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Correlation Heatmap of Numerical Variables')
plt.show()

**Correlation Analysis (Heatmap)**

**Price Multicollinearity:** The variables Open, High, Low, and Close show a near-perfect positive correlation (~1.0). For modeling, tracking just the Close price is sufficient to understand the stock's trend, as the others contain redundant information.

**Sentiment vs. Price:** We observe a weak positive correlation (0.14) between Label (Sentiment) and Close price. This proves that market sentiment does not drive stock prices in a simple, linear way. A "Positive" headline does not guarantee a proportional price increase on the exact same day, justifying the need for non-linear models like Neural Networks later in this project.

**Volume vs. Price:** We observe a weak negative correlation (-0.10) between Volume and Close price. This indicates a slight tendency for trading volume to increase when the stock price decreases, aligning with "panic selling" behavior.

In [ ]:
# --- 2. Sentiment Polarity vs Price (Boxplot) ---

plt.figure(figsize=(10, 6))
sns.boxplot(x='Label', y='Close', data=data, palette='Set2')
plt.title('Stock Closing Price by Sentiment Polarity')
plt.xlabel('Sentiment (-1: Negative, 0: Neutral, 1: Positive)')
plt.ylabel('Closing Price ($)')
plt.show()

**Observation:** The median closing price is roughly similar across Positive (1) and Negative (-1) sentiment days.

**Insight:** This confirms the heatmap findings. Sentiment is a contributing factor rather than a sole determinant of daily price. The broader market may have already "priced in" the news before the headline appeared, or macro-economic factors are carrying the stock's momentum regardless of daily news.

In [ ]:
# --- 3. Date vs Price (Time Series Line Plot) ---

plt.figure(figsize=(14, 6))
# It's good practice to ensure the Date column is actually a datetime object for plotting
if not pd.api.types.is_datetime64_any_dtype(data['Date']):
    data['Date'] = pd.to_datetime(data['Date'])

sns.lineplot(x='Date', y='Close', data=data, color='blue', linewidth=1.5)
plt.title('Stock Closing Price Over Time')
plt.xlabel('Date')
plt.ylabel('Closing Price ($)')
plt.xticks(rotation=45) # Rotates the dates so they don't overlap
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

**Date vs. Price (Time Series)**

**Observation:** The stock shows clear directional trends and periods of distinct volatility over time, rather than a flat or entirely random walk.

**Insight:** Stock prices have a temporal dependency (momentum). A sudden drop in the line chart likely aligns with specific negative news events, but the overall trend dictates the baseline price.

In [ ]:
# --- 4. Sentiment vs Volume (Bar Chart checking for Panic Selling) ---
plt.figure(figsize=(10, 6))
sns.barplot(x='Label', y='Volume', data=data, palette='Reds', errorbar=None)
plt.title('Average Trading Volume by Sentiment')
plt.xlabel('Sentiment (-1: Negative, 0: Neutral, 1: Positive)')
plt.ylabel('Average Volume')
plt.show()

**Sentiment vs. Volume (The "Panic" Signal):** By plotting average volume against sentiment labels, we typically see that Negative (-1) news days trigger higher average trading volume than Neutral days. This confirms that bad news triggers a stronger immediate behavioral reaction (risk-off selling) than neutral updates.

In [ ]:
# --- 5.Sentiment vs News Length (Boxplot) --
# Safety check: ensure news_length exists just in case
if 'news_length' not in data.columns:
    data['news_length'] = data['News'].astype(str).apply(len)

plt.figure(figsize=(10, 6))
sns.boxplot(x='Label', y='news_length', data=data, palette='Blues')
plt.title('News Article Length by Sentiment')
plt.xlabel('Sentiment (-1: Negative, 0: Neutral, 1: Positive)')
plt.ylabel('News Length (Characters)')
plt.show()

**Sentiment vs. News Length:** Analyzing the length of the text across different sentiments reveals that article length remains relatively consistent. A highly negative event isn't necessarily a longer article than a positive one, meaning we don't need to treat text length as a proxy for sentiment severity.

## **Data Preprocessing**

In [ ]:
# --- 1. Split the target variable and predictors ---
# X is our input feature (the text we want the model to read)
# y is our target (the sentiment we want the model to predict)
X = data['News']
y = data['Label']

# --- 2. Split the data into train and test sets ---
# We allocate 20% of the data for testing and 80% for training.
# CRITICAL: 'stratify=y' ensures the minority classes (like Neutral) are distributed proportionally.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# --- 3. Verify the splits ---
print(f"Training data shape: X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"Testing data shape: X_test: {X_test.shape}, y_test: {y_test.shape}")

print("\n--- Class Distribution in Train Set ---")
print(y_train.value_counts())

print("\n--- Class Distribution in Test Set ---")
print(y_test.value_counts())

**Data Splitting Strategy:**
We isolated the News column as our single predictor (X) and Label as our target (y). We applied an 80/20 train-test split. Crucially, because our exploratory data analysis revealed a severe class imbalance (especially regarding the 'Neutral' class), we applied Stratified Sampling (stratify=y). This guarantees that the exact ratio of Positive, Negative, and Neutral labels is perfectly maintained across both the training and testing datasets, preventing skewed evaluation results.

## **Word Embeddings**

#### **Word2Vec**

In [ ]:

print("--- Generating Word2Vec Embeddings ---")

vec_size = 300
# Train the Word2Vec Model on the training data ONLY
words_list_train = [str(item).split() for item in X_train.values]
word2vec_model = Word2Vec(words_list_train, vector_size=vec_size, window=5, min_count=1, workers=6)

# Checking the size of the vocabulary
print("Length of the vocabulary is", len(list(word2vec_model.wv.key_to_index)))


# Retrieving the words present in the Word2Vec model's vocabulary
words = list(word2vec_model.wv.key_to_index.keys())

# Retrieving word vectors for all the words present in the model's vocabulary
wvs = word2vec_model.wv[words].tolist()

# Creating a dictionary of words and their corresponding vectors
word_vector_dict = dict(zip(words, wvs))


# Your custom vectorizer function
def average_vectorizer_Word2Vec(doc):
    # Initializing a feature vector for the sentence
    feature_vector = np.zeros((vec_size,), dtype="float64")

    # Creating a list of words in the sentence that are present in the model vocabulary
    words_in_vocab = [word for word in doc.split() if word in words]

    # adding the vector representations of the words
    for word in words_in_vocab:
        feature_vector += np.array(word_vector_dict[word])

    # Dividing by the number of words to get the average vector
    if len(words_in_vocab) != 0:
        feature_vector /= len(words_in_vocab)

    return feature_vector


# Apply to Train and Test sets
X_train_w2v = pd.DataFrame(X_train.apply(average_vectorizer_Word2Vec).tolist(),
                           columns=['W2V_Feature_'+str(i) for i in range(vec_size)])
X_test_w2v = pd.DataFrame(X_test.apply(average_vectorizer_Word2Vec).tolist(),
                          columns=['W2V_Feature_'+str(i) for i in range(vec_size)])

print(f"Word2Vec Train Shape: {X_train_w2v.shape}")
print(f"Word2Vec Test Shape: {X_test_w2v.shape}\n")

### **Observation: Word2Vec Embeddings**
* **Vocabulary Learned:** The Word2Vec model successfully built a vocabulary of 13,001 unique words exclusively from the training data, preventing any data leakage from the test set.
* **Feature Dimensions:** We manually set the hyperparameter `vector_size=300`. Consequently, our custom averaging function transformed each news article into a dense vector of exactly 300 numerical features.
* **Technical Insight:** Because we had to mathematically average the individual word vectors to create a single document vector, the sequential context and grammar of the sentences are lost. The model knows *which* words are present, but not their specific order.

### **Sentence Transformer**

In [ ]:
print("--- Generating Sentence Transformer Embeddings ---")

# 1. Load the pre-trained Sentence Transformer model
# 'all-MiniLM-L6-v2' is an industry-standard, lightweight model for sentence embeddings
st_model = SentenceTransformer('all-MiniLM-L6-v2')

# 2. Encode the text directly into mathematical vectors
# We convert X_train and X_test to strings and lists, then pass them to .encode()
# show_progress_bar=True will give you a nice loading bar while it processes!
st_train_embeddings = st_model.encode(X_train.astype(str).tolist(), show_progress_bar=True)
st_test_embeddings = st_model.encode(X_test.astype(str).tolist(), show_progress_bar=True)

# 3. Convert the output arrays into Pandas DataFrames (matching our Word2Vec format)
# This model naturally outputs 384 features per sentence
X_train_st = pd.DataFrame(st_train_embeddings,
                          columns=['ST_Feature_'+str(i) for i in range(st_train_embeddings.shape[1])])

X_test_st = pd.DataFrame(st_test_embeddings,
                         columns=['ST_Feature_'+str(i) for i in range(st_test_embeddings.shape[1])])

# 4. Display the final shapes
print(f"Sentence Transformer Train Shape: {X_train_st.shape}")
print(f"Sentence Transformer Test Shape: {X_test_st.shape}")

### **Observation: Sentence Transformer Embeddings**
* **Feature Dimensions:** The pre-trained `all-MiniLM-L6-v2` model naturally outputs vectors with a hardcoded dimension of 384. Therefore, our resulting datasets (`X_train_st` and `X_test_st`) contain 384 features per article.
* **Technical Insight:** Unlike Word2Vec's "bag-of-words" averaging approach, the Sentence Transformer uses self-attention to read the entire sentence dynamically. It preserves word order, grammar, and financial context (e.g., distinguishing between a "river bank" and a "Wall Street bank").
* **Expectation:** Because financial news heavily relies on nuanced context, we expect the 384-dimensional Sentence Transformer embeddings to yield higher accuracy in the upcoming classification models than the Word2Vec embeddings.

## **Sentiment Analysis**

### **Model Evaluation Criterion**

**Note:**  
You can use the helper functions provided below to:
- Plot a **confusion matrix** (`plot_confusion_matrix`)
- Generate key **classification metrics** like accuracy, recall, precision, and F1-score (`model_performance_classification_sklearn`)

These are ready-to-use. However, you’re welcome to explore and write your own evaluation code if you prefer. Feel free to modify or extend these as per your learning goals!

##### **Utility Functions**

In [ ]:
def plot_confusion_matrix(actual, predicted):
    """
    Plot a confusion matrix to visualize the performance of a classification model.

    Parameters:
    actual (array-like): The true labels.
    predicted (array-like): The predicted labels from the model.

    Returns:
    None: Displays the confusion matrix plot.
    """

    # Compute the confusion matrix.
    cm = confusion_matrix(actual, predicted)

    # Create a new figure with a specified size
    plt.figure(figsize=(5, 4))

    # Define the labels for the confusion matrix dynamically from the data
    #label_list = sorted(list(np.unique(np.concatenate((actual, predicted)))))
    label_list = ['Negative', 'Neutral', 'Positive']

    # Plot the confusion matrix using a heatmap with annotations
    sns.heatmap(cm, annot=True, fmt='.0f', cmap='Blues', xticklabels=label_list, yticklabels=label_list)

    # Label for the y-axis
    plt.ylabel('Actual')

    # Label for the x-axis
    plt.xlabel('Predicted')

    # Title of the plot
    plt.title('Confusion Matrix')

    # Display the plot
    plt.show()

In [ ]:
def model_performance_classification_sklearn(actual, predicted):
    """
    Compute various performance metrics for a classification model using sklearn.

    Parameters:
    model (sklearn classifier): The classification model to evaluate.
    predictors (array-like): The independent variables used for predictions.
    target (array-like): The true labels for the dependent variable.

    Returns:
    pandas.DataFrame: A DataFrame containing the computed metrics (Accuracy, Recall, Precision, F1-score).
    """

    # Compute Accuracy
    acc = accuracy_score(actual,predicted)
    # Compute Recall
    recall = recall_score(actual,predicted,average='weighted')
    # Compute Precision
    precision = precision_score(actual,predicted,average='weighted')
    # Compute F1-score
    f1 = f1_score(actual,predicted,average='weighted')

    # Create a DataFrame to store the computed metrics
    df_perf = pd.DataFrame(
        {
            "Accuracy": [acc],
            "Recall": [recall],
            "Precision": [precision],
            "F1": [f1],
        }
    )
    # Return the DataFrame with the metrics
    return df_perf

##### **Build Random Forest Models using different text embeddings**

###### **RANDOM FOREST WITH WORD2VEC**

In [ ]:
print("==============================================")
print("  MODEL 1: RANDOM FOREST WITH WORD2VEC")
print("==============================================\n")

# ==========================================
# 1. Train the model with n_estimators=200, max_depth=8
# ==========================================
rf_v1 = RandomForestClassifier(n_estimators=100, max_depth=8,
                               class_weight='balanced', random_state=42, n_jobs=-1)
rf_v1.fit(X_train_w2v, y_train)

v1_train_preds = rf_v1.predict(X_train_w2v)
v1_test_preds = rf_v1.predict(X_test_w2v)

v1_train_perf = model_performance_classification_sklearn(y_train, v1_train_preds)
v1_test_perf = model_performance_classification_sklearn(y_test, v1_test_preds)
v1_summary = pd.concat([v1_train_perf, v1_test_perf])
v1_summary.index = ['Training Data', 'Testing Data']

# ==========================================
# 2. TUNED VERSION with n_estimators=300, max_depth=4, min_samples_leaf=5
# ==========================================
rf_v2 = RandomForestClassifier(n_estimators=300, max_depth=4, min_samples_leaf=5,
                               class_weight='balanced', random_state=42, n_jobs=-1)
rf_v2.fit(X_train_w2v, y_train)

v2_train_preds = rf_v2.predict(X_train_w2v)
v2_test_preds = rf_v2.predict(X_test_w2v)

v2_train_perf = model_performance_classification_sklearn(y_train, v2_train_preds)
v2_test_perf = model_performance_classification_sklearn(y_test, v2_test_preds)
v2_summary = pd.concat([v2_train_perf, v2_test_perf])
v2_summary.index = ['Training Data', 'Testing Data']


# ==========================================
# 3. SIDE-BY-SIDE MODEL PERFORMANCE DISPLAY
# ==========================================
print("--- WORD2VEC PERFORMANCE COMPARISON ---")
combined_w2v = pd.concat([v1_summary, v2_summary], axis=1,
                         keys=['V1: Est=200, Depth=8', 'V2: Est=300, Depth=4, min_samples_leaf=5'])
display(combined_w2v)

#print("\n--- V2 (Depth=8): TEST SET CONFUSION MATRIX ---")
#plot_confusion_matrix(y_test, v2_test_preds)

# ==========================================
# 4. CONFUSION MATRICES (V1 vs V2)
# ==========================================
print("\n" + "="*50)
print("V1: DEPTH 8 - CONFUSION MATRICES")
print("="*50)
print("--- V1: TRAINING SET ---")
plot_confusion_matrix(y_train, v1_train_preds)
print("\n--- V1: TEST SET ---")
plot_confusion_matrix(y_test, v1_test_preds)

print("\n" + "="*50)
print("V2: DEPTH 4 (TUNED) - CONFUSION MATRICES")
print("="*50)
print("--- V2: TRAINING SET ---")
plot_confusion_matrix(y_train, v2_train_preds)
print("\n--- V2: TEST SET ---")
plot_confusion_matrix(y_test, v2_test_preds)

**Observation: Bias-Variance Optimization**

**Taming Overfitting:** By moving from V1 (Depth 8) to V2 (Depth 4), we successfully mitigated severe overfitting. While the training F1-score dropped from ~0.99 to ~0.78, the test F1-score remained highly stable (shifting only slightly from 0.578 to 0.570). This indicates that the simpler V2 model successfully closed the massive generalization gap and is no longer just "memorizing" the training data.

**Precision Gains:** Interestingly, the Testing Precision for V2 (0.584) is higher than V1 (0.565). In a stock market context, this is critical—it means when the heavily regularized V2 model predicts a sentiment, it is more likely to be correct, reducing the risk of false positives.

**Algorithmic Plateau:** Even with hyperparameter tuning, both models struggle to break past the ~60% mark on unseen testing data. This suggests that the "bag-of-words" averaging approach in Word2Vec may be losing critical linguistic context that a Decision Tree simply cannot recover, paving the way for transformer-based embeddings.

###### **RANDOM FOREST WITH Sentence Transformers**

In [ ]:
# ==========================================
# 1. Train the model with n_estimators=200, max_depth=8
# ==========================================
rf_st_v1 = RandomForestClassifier(n_estimators=200, max_depth=8,
                               class_weight='balanced', random_state=42, n_jobs=-1)
rf_st_v1.fit(X_train_st, y_train)

v1_st_train_preds = rf_st_v1.predict(X_train_st)
v1_st_test_preds = rf_st_v1.predict(X_test_st)

v1_st_train_perf = model_performance_classification_sklearn(y_train, v1_st_train_preds)
v1_st_test_perf = model_performance_classification_sklearn(y_test, v1_st_test_preds)
v1_st_summary = pd.concat([v1_st_train_perf, v1_st_test_perf])
v1_st_summary.index = ['Training Data', 'Testing Data']

# ==========================================
# 2. TUNED VERSION with n_estimators=300, max_depth=4, min_samples_leaf=5
# ==========================================
rf_st_v2 = RandomForestClassifier(n_estimators=300, max_depth=4, min_samples_leaf=5,
                               class_weight='balanced', random_state=42, n_jobs=-1)
rf_st_v2.fit(X_train_st, y_train)

v2_st_train_preds = rf_st_v2.predict(X_train_st)
v2_st_test_preds = rf_st_v2.predict(X_test_st)

v2_st_train_perf = model_performance_classification_sklearn(y_train, v2_st_train_preds)
v2_st_test_perf = model_performance_classification_sklearn(y_test, v2_st_test_preds)
v2_st_summary = pd.concat([v2_st_train_perf, v2_st_test_perf])
v2_st_summary.index = ['Training Data', 'Testing Data']


# ==========================================
# 3. SIDE-BY-SIDE MODEL PERFORMANCE DISPLAY
# ==========================================
print("--- SENTENCE TRANSFORMERS PERFORMANCE COMPARISON ---")
combined_st = pd.concat([v1_st_summary, v2_st_summary], axis=1,
                         keys=['V1: Est=200, Depth=8', 'V2: Est=300, Depth=4, min_samples_leaf=5'])
display(combined_st)

# ==========================================
# 4. CONFUSION MATRICES (V1 vs V2)
# ==========================================
print("\n" + "="*50)
print("V1: DEPTH 8 - CONFUSION MATRICES")
print("="*50)
print("--- V1: TRAINING SET ---")
plot_confusion_matrix(y_train, v1_st_train_preds)
print("\n--- V1: TEST SET ---")
plot_confusion_matrix(y_test, v1_st_test_preds)

print("\n" + "="*50)
print("V2: DEPTH 4 (TUNED) - CONFUSION MATRICES")
print("="*50)
print("--- V2: TRAINING SET ---")
plot_confusion_matrix(y_train, v2_st_train_preds)
print("\n--- V2: TEST SET ---")
plot_confusion_matrix(y_test, v2_st_test_preds)

**Observations: Random Forest with Sentence Transformers**

* **The Transformer Advantage:** The Sentence Transformer embeddings completely outperformed the Word2Vec embeddings. The tuned Testing F1-Score jumped from **~0.57** (Word2Vec) to **~0.67** (Sentence Transformers). This happens because Word2Vec averages words in isolation, losing the structure of the sentence. Sentence Transformers (based on BERT architecture) read the entire headline contextually, allowing the model to grasp subtle financial sentiment much better.

* **Overfitting & Tuning Success:** Just like the Word2Vec model, **V1 (Depth 8)** severely overfit the data, scoring a perfect 1.00 (100%) on the training set but dropping to 0.63 on the test set. By restricting the model in **V2 (Depth 4, min_samples_leaf=5)**, we brought the training score down to a more realistic 0.92, which allowed the real-world Testing F1 to rise to **0.67**.
* **The "Curse of Dimensionality":** While 0.67 is a massive improvement, we can see that the model still has a noticeable generalization gap (0.92 Train vs 0.67 Test). Random Forests are built to make perpendicular cuts on tabular data. Forcing a decision tree to process 384 columns of dense, continuous neural embeddings is mathematically inefficient.

**Next Steps:** To fully unlock the potential of these rich 384-dimensional embeddings, we need to use an algorithm natively designed to process them: a Neural Network (Multi-Layer Perceptron).

### **Building Neural Network Models using different text embeddings**

###### **Neural Network WITH WORD2VEC**

In [ ]:
print("==============================================")
print("  MODEL 3: KERAS NEURAL NETWORK WITH WORD2VEC")
print("==============================================\n")

# 1. Keras Label Encoding (Safe conversion to 0, 1, 2)
le = LabelEncoder()
y_train_nn = le.fit_transform(y_train)
y_test_nn = le.transform(y_test)

# Number of features (columns) in Word2Vec
input_dim_w2v = X_train_w2v.shape[1]

# ==========================================
# VERSION 1: Deep Network (128 and 64 neurons)
# ==========================================
tf.keras.backend.clear_session()
gc.collect()

nn_v1 = Sequential([
    # First hidden layer: 128 neurons, using tanh!
    Dense(128, activation='tanh', input_shape=(input_dim_w2v,)),
    Dropout(0.1),

    # Second hidden layer: 64 neurons, using tanh!
    Dense(64, activation='tanh'),

    # Output layer: MUST be softmax for 3-class prediction
    Dense(3, activation='softmax')
])

nn_v1.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
nn_v1.fit(X_train_w2v, y_train_nn, epochs=20, batch_size=16, verbose=0)

v1_nn_train_preds = le.inverse_transform(np.argmax(nn_v1.predict(X_train_w2v, verbose=0), axis=1))
v1_nn_test_preds = le.inverse_transform(np.argmax(nn_v1.predict(X_test_w2v, verbose=0), axis=1))

v1_nn_train_perf = model_performance_classification_sklearn(y_train, v1_nn_train_preds)
v1_nn_test_perf = model_performance_classification_sklearn(y_test, v1_nn_test_preds)
v1_nn_summary = pd.concat([v1_nn_train_perf, v1_nn_test_perf])
v1_nn_summary.index = ['Training Data', 'Testing Data']

# ==========================================
# VERSION 2: Tuned Network (32 neurons)
# ==========================================
tf.keras.backend.clear_session()
gc.collect()

nn_v2 = Sequential([
    # Single hidden layer: 32 neurons, using tanh AND L2 Regularization!
    Dense(32, activation='tanh', kernel_regularizer=regularizers.l2(0.05), input_shape=(input_dim_w2v,)),

    # Output layer: MUST be softmax
    Dense(3, activation='softmax')
])

nn_v2.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
nn_v2.fit(X_train_w2v, y_train_nn, epochs=20, batch_size=16, verbose=0)

v2_nn_train_preds = le.inverse_transform(np.argmax(nn_v2.predict(X_train_w2v, verbose=0), axis=1))
v2_nn_test_preds = le.inverse_transform(np.argmax(nn_v2.predict(X_test_w2v, verbose=0), axis=1))

v2_nn_train_perf = model_performance_classification_sklearn(y_train, v2_nn_train_preds)
v2_nn_test_perf = model_performance_classification_sklearn(y_test, v2_nn_test_preds)
v2_nn_summary = pd.concat([v2_nn_train_perf, v2_nn_test_perf])
v2_nn_summary.index = ['Training Data', 'Testing Data']

# ==========================================
# SIDE-BY-SIDE DISPLAY
# ==========================================
print("--- WORD2VEC: KERAS NEURAL NETWORK COMPARISON ---")
combined_nn_w2v = pd.concat([v1_nn_summary, v2_nn_summary], axis=1,
                         keys=['V1: Hidden(128,64), Dropout', 'V2: Hidden(32), L2, Dropout'])
display(combined_nn_w2v)


# ==========================================
# CONFUSION MATRICES
# ==========================================
print("\n" + "="*50)
print("V1: DEEP NETWORK (128, 64) - CONFUSION MATRICES")
print("="*50)
print("--- V1: TRAINING SET ---")
plot_confusion_matrix(y_train, v1_nn_train_preds)
print("\n--- V1: TEST SET ---")
plot_confusion_matrix(y_test, v1_nn_test_preds)

print("\n" + "="*50)
print("V2: TUNED NETWORK (32) - CONFUSION MATRICES")
print("="*50)
print("--- V2: TRAINING SET ---")
plot_confusion_matrix(y_train, v2_nn_train_preds)
print("\n--- V2: TEST SET ---")
plot_confusion_matrix(y_test, v2_nn_test_preds)

**Observations: Keras Neural Network with Word2Vec**

* **A Classic Case of Underfitting:** Despite optimizing the network to allow maximum learning capacity (reducing Dropout in V1 to 0.1 and removing it entirely in V2), the model's performance hit a hard ceiling. Both the Training F1 (0.57 max) and Testing F1 (0.51 max) are incredibly close, meaning the model is not overfitting, but it is severely underfitting.
* **The "Glass Ceiling" of Word2Vec:** The neural network simply cannot find meaningful mathematical boundaries in the 300-dimensional Word2Vec space. Because Word2Vec averages out word vectors, it loses the sequential context of the financial headlines. Without tens of thousands of examples to brute-force a pattern, the neural network struggles to map these flat embeddings to accurate sentiment categories.
* **Architecture vs. Algorithm:** Interestingly, the simpler Random Forest model actually outperformed the Neural Network on this Word2Vec data (Test F1 of ~0.57 vs ~0.51). Neural networks require highly complex, non-linear patterns to thrive; when the data representation is too simple or context-poor, a basic decision tree is often more effective.

###### **Neural Network WITH Sentence Tranformers**

In [ ]:
print("==============================================")
print("  MODEL 4: NEURAL NETWORK WITH SENTENCE TRANSFORMERS")
print("==============================================\n")

# Number of features (columns) in Sentence Transformers (usually 384)
input_dim_st = X_train_st.shape[1]

# ==========================================
# VERSION 1: Deep Network (128 and 64 neurons)
# ==========================================
tf.keras.backend.clear_session()
gc.collect()

nn_st_v1 = Sequential([
    Dense(64, activation='tanh', input_shape=(input_dim_st,)),
    Dropout(0.5),
    Dense(32, activation='tanh'),
    Dense(3, activation='softmax')
])

nn_st_v1.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
nn_st_v1.fit(X_train_st, y_train_nn, epochs=20, batch_size=16, verbose=0)

v1_nn_st_train_preds = le.inverse_transform(np.argmax(nn_st_v1.predict(X_train_st, verbose=0), axis=1))
v1_nn_st_test_preds = le.inverse_transform(np.argmax(nn_st_v1.predict(X_test_st, verbose=0), axis=1))

v1_nn_st_train_perf = model_performance_classification_sklearn(y_train, v1_nn_st_train_preds)
v1_nn_st_test_perf = model_performance_classification_sklearn(y_test, v1_nn_st_test_preds)
v1_nn_st_summary = pd.concat([v1_nn_st_train_perf, v1_nn_st_test_perf])
v1_nn_st_summary.index = ['Training Data', 'Testing Data']

# ==========================================
# VERSION 2: Tuned Network (32 neurons)
# ==========================================
tf.keras.backend.clear_session()
gc.collect()

nn_st_v2 = Sequential([
    Dense(32, activation='tanh',
          #kernel_regularizer=regularizers.l2(0.05),
          input_shape=(input_dim_st,)),
    Dropout(0.5),
    Dense(3, activation='softmax')
])

nn_st_v2.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
nn_st_v2.fit(X_train_st, y_train_nn, epochs=20, batch_size=16, verbose=0)

v2_nn_st_train_preds = le.inverse_transform(np.argmax(nn_st_v2.predict(X_train_st, verbose=0), axis=1))
v2_nn_st_test_preds = le.inverse_transform(np.argmax(nn_st_v2.predict(X_test_st, verbose=0), axis=1))

v2_nn_st_train_perf = model_performance_classification_sklearn(y_train, v2_nn_st_train_preds)
v2_nn_st_test_perf = model_performance_classification_sklearn(y_test, v2_nn_st_test_preds)
v2_nn_st_summary = pd.concat([v2_nn_st_train_perf, v2_nn_st_test_perf])
v2_nn_st_summary.index = ['Training Data', 'Testing Data']

# ==========================================
# SIDE-BY-SIDE DISPLAY
# ==========================================
print("--- SENTENCE TRANSFORMERS: KERAS NEURAL NETWORK COMPARISON ---")
combined_nn_st = pd.concat([v1_nn_st_summary, v2_nn_st_summary], axis=1,
                         keys=['V1: Hidden(128,64), Dropout', 'V2: Hidden(32), L2, Dropout'])
display(combined_nn_st)

# ==========================================
# CONFUSION MATRICES
# ==========================================
print("\n" + "="*50)
print("V1: DEEP NETWORK (128, 64) - CONFUSION MATRICES")
print("="*50)
print("--- V1: TRAINING SET ---")
plot_confusion_matrix(y_train, v1_nn_st_train_preds)
print("\n--- V1: TEST SET ---")
plot_confusion_matrix(y_test, v1_nn_st_test_preds)

print("\n" + "="*50)
print("V2: TUNED NETWORK (32) - CONFUSION MATRICES")
print("="*50)
print("--- V2: TRAINING SET ---")
plot_confusion_matrix(y_train, v2_nn_st_train_preds)
print("\n--- V2: TEST SET ---")
plot_confusion_matrix(y_test, v2_nn_st_test_preds)

**Observations: Neural Network with Sentence Transformers**

**The Capacity Advantage (V1):** By utilizing a deeper architecture (64 and 32 neurons) with a 0.5 Dropout rate, the model successfully captured the complex, high-dimensional patterns of the Transformer embeddings. It achieved a Training F1 of ~0.89 and the highest Testing F1 of ~0.71 (with a Testing Accuracy of 72.6%). The 50% Dropout successfully prevented total memorization, keeping the generalization gap at an acceptable 18 points while delivering the project's peak predictive power.

**The Constraint of Simplicity (V2):** The single-layer 32-neuron model proved to be slightly too constrained in this iteration. While it maintained a tighter generalization gap (Training F1 ~0.81 vs Testing F1 ~0.67), its overall real-world accuracy dropped to 70.2%. This suggests that 32 neurons alone may not possess quite enough "learning capacity" to fully decode the rich semantic signals provided by the 384-dimensional Sentence Transformers.

**Optimal Architecture Selection:** These metrics confirm that the dual-layer architecture (V1) provides the best overall balance for this specific dataset. It utilizes enough neurons to understand the complex financial context, while relying heavily on Dropout to prevent the severe overfitting seen in earlier tree-based models.

### **Model Performance Summary and Final Model Selection**

In [ ]:
print("=================================================================")
print("               FINAL MODEL PERFORMANCE SUMMARY")
print("=================================================================\n")

# Combine the best performing or tuned dataframes from each of your 4 sections
final_summary = pd.concat(
    [
        v2_summary,              # Word2Vec + RF (Tuned)
        v2_st_summary,           # Sentence Transformer + RF (Tuned)
        v2_nn_summary,           # Word2Vec + NN (Tuned)
        v1_nn_st_summary         # Sentence Transformer + NN (CHAMPION - V1)
    ],
    axis=1,
    keys=[
        'Word2Vec-RF',
        'Sentence-Transformer-RF',
        'Word2Vec-NN',
        'Sentence-Transformer-NN'
    ]
)

display(final_summary)

**Final Observations**

**Word2Vec with Random Forest (Word2Vec-RF):**
By restricting tree depth, we achieved a healthy generalization gap (0.79 Train F1 vs. 0.57 Test F1), but the performance ceiling remains low. This confirms that Word2Vec’s "flat" word-averaging approach is insufficient for the high-stakes nuance of financial sentiment, where a single word can flip the entire meaning of a headline.

**Sentence Transformer with Random Forest (Sentence-Transformer-RF):**
This model represents a massive leap in quality, with the Testing F1-Score jumping to 0.67. While Sentence Transformers provide the rich, semantic context necessary for better classification, this model exhibits significant overfitting (0.93 Train vs. 0.67 Test). This indicates that tree-based models struggle to generalize when faced with the high-dimensional density of Transformer embeddings on a small dataset.

**Word2Vec with Neural Network (Word2Vec-NN):**
This approach resulted in the most significant underfitting. With a Testing F1 of only 0.50, the nearly identical Train and Test scores show that while the model is "stable," it is not actually learning. A neural network simply cannot find enough signal in 300 "flat" Word2Vec dimensions when limited to only ~334 training samples.

**Sentence Transformer with Neural Network (Sentence-Transformer-NN):**
This is the project's breakthrough model. By utilizing a dual-layer architecture (64 and 32 neurons) and 50% Dropout, we achieved the highest real-world performance with a Testing F1-Score of ~0.71 and a Testing Accuracy of 72.6%. Most importantly, we successfully addressed the extreme overfitting issues seen in tree-based models, utilizing Dropout to maintain a solid balance between learning complex semantic signals and generalizing to unseen data (0.89 Train vs. 0.71 Test).

**Final Model Selection:** Sentence Transformer with Neural Network

**Why I chose this model:**
After evaluating four distinct approaches through rigorous hyperparameter tuning, the Sentence-Transformer-NN (64/32-Neuron, 0.5 Dropout) is definitively the superior model for predicting financial news sentiment.

**Proper Reasoning:**

**Highest Generalization Power:** This model achieved the most reliable metrics on unseen data, including a Testing F1-Score of 0.71. Unlike the Random Forest version, which severely "memorized" the training set, this Neural Network maintains a healthy balance between learning complex patterns and generalizing.

**Superior Contextual Utility: **The results prove that BERT-based Sentence Transformers are the "gold standard" for this task. The neural network successfully decoded the 384-dimensional semantic space to understand subtle financial tones that Word2Vec missed entirely.

**Stability in Production:** With a 72.6% Testing Accuracy, this model is the most dependable choice for a real-world financial environment. The use of a dual-layer setup provided the "brainpower" to understand the data, while the 50% Dropout ensured that the model remained robust enough to handle new, unseen headlines without falling into the "Memorization Trap."

## **Conclusions and Recommendations**

**Conclusions**

1. **The Superiority of Context:** This project successfully demonstrates that advanced, context-aware embeddings are critical for financial sentiment analysis. While traditional Word2Vec models struggled to capture the nuance of headlines, Sentence Transformers (BERT-based) provided a much richer semantic foundation, allowing models to cross the threshold of meaningful predictive power.

2. **Model Architecture:** The Neural Network (dual-layer 64/32-neuron architecture) emerged as the champion over Random Forest models. While tree-based models were prone to extreme overfitting on high-dimensional transformer embeddings, the regularized Neural Network achieved the best balance of learning capacity and generalization, proving its reliability for real-world application.

3. **Generalization Mastery:** By employing a high Dropout rate (50%) alongside the dual-layer architecture, the project successfully addressed the "Memorization Trap." The final model maintained a healthy 18-point generalization gap (Training: ~0.89 F1 vs. Testing: ~0.71 F1), ensuring that the performance seen during training will translate to actual unseen market news.

**Actionable Insights**

1. **Semantics Over Frequency:** The success of the Transformer-NN pipeline proves that the "context" of a financial headline is far more important than the simple frequency of keywords. The model can identify sentiment shifts even when standard "bullish" or "bearish" terms are absent.

2. **The Imbalance Challenge:** Analysis of the results reveals lower recall for the Neutral sentiment category. This is identified as a result of the extreme class imbalance (only 1.7% of the data) rather than a model failure.

3. **Predictive Stability:** The consistency between training and testing metrics in the champion model indicates that the feature set provided by the Sentence Transformers is stable and resilient to the inherent noise found in financial media.

**Business Recommendations**

1. **Automated Sentiment Filtering:** The startup should integrate the Sentence-Transformer-NN model into an automated news-monitoring pipeline. This system can scan thousands of daily headlines, flagging only the highest-confidence "Positive" or "Negative" signals for immediate analyst review, potentially increasing operational efficiency by over 50%.

2. **Targeted Data Acquisition:** To address the performance gap in neutral sentiment, the firm should initiate a "Balanced Labeling Project." Adding 200-300 samples of neutral financial news will improve the model’s ability to distinguish between "no news" and "market-moving news."

3. **Signal Integration:** The sentiment scores generated by this model should not be used in isolation. Instead, they should be integrated as a "Sentiment Feature" into broader quantitative trading algorithms to capture the human-behavioral element of market volatility.

4. **Continuous Learning Loop:** Implement a feedback loop where human analysts can verify or correct the model's predictions. This "Human-in-the-Loop" data can be used for periodic model retraining, ensuring the AI stays updated with evolving financial terminology and market cycles.

<font size=6 color='blue'>Power Ahead</font>
___